![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 2 · Actividad calificada · aprendizaje autónomo</div><div style="font-size:22px;font-weight:700;margin-top:4px">Deber 3 · Ruta A · Un flujo que escala con Spark MLlib</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Qué entregas** | **Un solo notebook ejecutado** (`.ipynb`): este (ruta A, Spark) **o** el de la ruta B (PyTorch). Renómbralo `D3_Apellido_Nombre.ipynb` y súbelo al aula virtual. No hay que subir nada más. |
| **Fecha** | **Jueves 19 de noviembre de 2026** |
| **Tiempo estimado** | 3 a 4 horas (Spark tarda unos 10 minutos en total en Colab) |
| **Peso** | 1/3 de los 2 puntos de aprendizaje autónomo · rúbrica al final de este notebook y en el aula |
| **Uso de IA** | **Permitido sin límite**: Claude Code, Codex, ChatGPT, Gemini en Colab, Copilot u otra, siempre que lo **declares**. Este deber incluye un **desafío con IA** obligatorio (parte 6). |
| **Teoría** | Manual M2 §10–§11 · laboratorio L2.4 (escalar con Spark y PyTorch) |
| **Datos** | **Fashion-MNIST completo**: 60 000 fotos de entrenamiento y 10 000 de prueba. La carga ya está hecha. |

**Qué vas a hacer.** Construir en **Spark MLlib** el mismo flujo que harías en scikit-learn (preparar → reducir → clasificar → validar), con los datos completos, y comparar exactitud y tiempo con scikit-learn. Spark es la herramienta de los *data lakes*: el mismo código corre en tu equipo o en un clúster de cientos de máquinas.

**Partes:** 0 · cómo trabajar → 1 · configuración y Spark → 2 · datos en formato LIBSVM (hecho) → 3 · Pipeline de Spark → 4 · validación cruzada con `CrossValidator` → 5 · comparación con scikit-learn → 6 · desafío con IA (script y SLURM) → 7 · reflexión → declaración y lista de control.

## 0 · Cómo trabajar este notebook

1. **Guarda tu copia.** En Colab: *Archivo → Guardar una copia en Drive*.
2. **Ejecuta de arriba hacia abajo.** La parte 1 instala `pyspark` si falta (tarda un minuto la primera vez). Spark necesita Java 17 o superior: Colab ya lo trae.
3. **Completa las celdas `# TODO`.** Cada parte explica qué se pide y trae una pista. Las celdas **✅ Autoverificación** te dicen si vas bien.
4. **Responde en las celdas ✍️** con tus palabras.
5. **Antes de entregar:** *Reiniciar y ejecutar todas*; luego *Archivo → Descargar → Descargar .ipynb*, renómbralo `D3_Apellido_Nombre.ipynb` y súbelo.

> **¿Spark no arranca?** Revisa con `!java -version` que haya Java 17 o 21. Si el problema sigue, puedes cambiar a la **ruta B (PyTorch)**: el deber vale lo mismo.

## 1 · Configuración y sesión de Spark

In [ ]:
import gzip
import importlib.util
import os
import re
import subprocess
import sys
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

SEED = 2026
CACHE = os.path.expanduser("~/.cache/ml-unibe")
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"

# En Colab puede faltar pyspark: se instala una versión compatible con el Java disponible.
if importlib.util.find_spec("pyspark") is None:
    salida = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    java = int(re.search(r'version "(?:1\.)?(\d+)', salida).group(1))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark" if java >= 17 else "pyspark<4"])

import pyspark
from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[*]").appName("ml-unibe-D3A")
         .config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {pyspark.__version__} · núcleos: {spark.sparkContext.defaultParallelism} · scikit-learn {sklearn.__version__} · semilla {SEED}")

## 2 · Los datos en formato LIBSVM (ya resuelto)

Spark lee formatos pensados para grandes volúmenes: **Parquet**, ORC, CSV particionados o **LIBSVM** (disperso: guarda solo los píxeles distintos de cero). Escribimos Fashion-MNIST **una sola vez** en LIBSVM y Spark lo lee como un `DataFrame` con dos columnas: `label` y `features`.

<details><summary><b>Ayuda: cómo leer tus propios datos en Spark</b></summary>

- **CSV:** `spark.read.csv("ruta/*.csv", header=True, inferSchema=True)`; luego `VectorAssembler(inputCols=[...], outputCol="features")` junta las columnas numéricas.
- **Parquet** (lo más común en un *data lake*): `spark.read.parquet("ruta/")`. Desde pandas: `df.to_parquet("datos.parquet")`.
- **Desde Google Drive:** monta Drive (`from google.colab import drive; drive.mount("/content/drive")`) y usa rutas como `/content/drive/MyDrive/...`.
</details>

In [ ]:
from sklearn.datasets import dump_svmlight_file


def descargar(nombre):
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(URL + nombre, ruta)
    return ruta


def cargar_fashion():
    def imagenes(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)

    def etiquetas(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)
    return (imagenes("train-images-idx3-ubyte.gz"), etiquetas("train-labels-idx1-ubyte.gz"),
            imagenes("t10k-images-idx3-ubyte.gz"), etiquetas("t10k-labels-idx1-ubyte.gz"))


X_train, y_train, X_test, y_test = cargar_fashion()
rutas = {"train": os.path.join(CACHE, "spark", "fashion_train.libsvm"), "test": os.path.join(CACHE, "spark", "fashion_test.libsvm")}
os.makedirs(os.path.dirname(rutas["train"]), exist_ok=True)
for parte, (Xp, yp) in {"train": (X_train, y_train), "test": (X_test, y_test)}.items():
    if not os.path.exists(rutas[parte]) or os.path.getsize(rutas[parte]) == 0:
        dump_svmlight_file(Xp.astype(np.float32), yp, rutas[parte], zero_based=False)
    print(parte, f"{os.path.getsize(rutas[parte]) / 1e6:.0f} MB")

tr = spark.read.format("libsvm").option("numFeatures", 784).load(rutas["train"]).repartition(8).cache()
te = spark.read.format("libsvm").option("numFeatures", 784).load(rutas["test"]).cache()
print("filas:", tr.count(), "+", te.count())
tr.show(3, truncate=60)

## 3 · Un `Pipeline` de Spark

En Spark MLlib, como en scikit-learn, la preparación va **dentro** del pipeline. Los nombres cambian un poco: cada etapa lee una columna (`inputCol`) y escribe otra (`outputCol`).

**Tu tarea:** arma un `Pipeline` con **al menos dos etapas de preparación** y un clasificador:

1. `MinMaxScaler` o `StandardScaler` de `features` → `escaladas`;
2. `PCA(k=50)` de `escaladas` → `pca`;
3. `LogisticRegression(featuresCol="pca", labelCol="label", maxIter=50, family="multinomial")`.

Ajústalo con `tr` y mide la exactitud en `te` con `MulticlassClassificationEvaluator`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import PCA as SparkPCA
from pyspark.ml.feature import MinMaxScaler, StandardScaler

evaluador = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
# TODO: define las etapas, ajusta el pipeline con tr y evalúa en te.
escala = ...                         # StandardScaler(inputCol="features", outputCol="escaladas", withMean=False)
pca = ...                            # SparkPCA(k=50, inputCol="escaladas", outputCol="pca")
logistica = ...
pipeline = Pipeline(stages=[...])
t0 = time.time()
modelo = pipeline.fit(tr)
t_spark = time.time() - t0
exactitud_spark = evaluador.evaluate(modelo.transform(te))
print(f"Spark: exactitud en la prueba = {exactitud_spark:.3f} · entrenamiento en {t_spark:.0f} s")

**✅ Autoverificación de la parte 3**

In [ ]:
assert len(pipeline.getStages()) >= 3, "El pipeline necesita al menos dos etapas de preparación y un clasificador."
assert exactitud_spark > 0.75, "La exactitud debería superar 0.75: revisa las columnas de entrada y salida."
print("✓ Parte 3 correcta")

## 4 · Ajustar un hiperparámetro con `CrossValidator`

**Tu tarea:** con `ParamGridBuilder`, prueba al menos **dos valores** de un hiperparámetro (por ejemplo, `regParam` de la logística: 0.0 y 0.01, o `k` del PCA: 30 y 50) y usa `CrossValidator` con **3 pliegues** sobre `tr`. Reporta la exactitud media de cada valor y evalúa el mejor modelo **una vez** en `te`.

**Pista:** `CrossValidator(estimator=pipeline, estimatorParamMaps=rejilla, evaluator=evaluador, numFolds=3, seed=SEED, parallelism=2)`. Esta celda tarda varios minutos.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# TODO: arma la rejilla y el CrossValidator, ajústalo con tr y evalúa el mejor modelo en te.
rejilla = ParamGridBuilder().addGrid(..., [...]).build()
cv = CrossValidator(...)
t0 = time.time()
modelo_cv = cv.fit(tr)
t_cv = time.time() - t0
for params, media in zip(rejilla, modelo_cv.avgMetrics):
    print({p.name: v for p, v in params.items()}, f"exactitud CV = {media:.3f}")
exactitud_cv = evaluador.evaluate(modelo_cv.bestModel.transform(te))
print(f"mejor modelo · prueba = {exactitud_cv:.3f} · validación cruzada en {t_cv:.0f} s")

**✅ Autoverificación de la parte 4**

In [ ]:
assert len(rejilla) >= 2, "Prueba al menos dos valores de un hiperparámetro."
assert len(modelo_cv.avgMetrics) == len(rejilla), "Cada combinación debe tener su exactitud media."
print("✓ Parte 4 correcta")

## 5 · El mismo modelo en scikit-learn

**Tu tarea:** arma en scikit-learn el **mismo** flujo (escalado → PCA de 50 → regresión logística multinomial) con los mismos datos completos, mide tiempo y exactitud, y completa la tabla.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression as SkLogistic
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler as SkScaler

# TODO: entrena el pipeline de scikit-learn con X_train / 255 y evalúa en X_test / 255.
sk = make_pipeline(...)
t0 = time.time()
sk.fit(...)
t_sk = time.time() - t0
exactitud_sk = sk.score(...)
comparacion = pd.DataFrame([
    {"herramienta": "Spark MLlib", "exactitud prueba": exactitud_spark, "tiempo (s)": t_spark},
    {"herramienta": "scikit-learn", "exactitud prueba": exactitud_sk, "tiempo (s)": t_sk},
])
comparacion.round(3)

✍️ **Pregunta 5.** ¿Cuál fue más rápido en tu equipo y por qué? ¿Las exactitudes se parecen? ¿Cuándo valdría la pena Spark aunque aquí sea más lento?

*Tu respuesta aquí.*

## 6 · Desafío con IA: script reproducible y guion de SLURM

El desafío obligatorio de este deber. Pide a un asistente de IA (Claude Code, Codex, ChatGPT, Gemini…) que convierta tu flujo en:

1. un **script de Python** (`entrenar_spark.py`) que reciba por argumentos la ruta de los datos y el número de núcleos, fije la semilla y guarde la exactitud en un archivo;
2. un **guion de SLURM** (`entrenar.slurm`) para lanzarlo en un clúster, con `[COMPLETAR]` en lo que dependa del centro de cómputo (partición, cuenta, módulos).

**Verifica** lo que te entrega: ¿el script corre aquí con `!python entrenar_spark.py ...`? ¿Da la misma exactitud que tu notebook? ¿El guion pide recursos razonables? Pega abajo la versión final (ya corregida por ti).

In [ ]:
# TODO: pega aquí el script final (puedes escribirlo a un archivo con %%writefile en otra celda) y el guion de SLURM.
SLURM = """..."""
print(SLURM)

## 7 · Reflexión (≈ 150 palabras)

✍️ ¿Qué cambiaría si los datos fueran **100 veces más** (6 millones de fotos, unos 50 GB)? Piensa en memoria, formato (Parquet), número de particiones, tiempo y costo, y en qué partes del código no tendrías que cambiar nada.

*Tu reflexión aquí.*

In [ ]:
spark.stop()     # libera la máquina virtual de Java

## Declaración de uso de IA

En este módulo **se permite el uso total de la IA** siempre que lo declares. Completa la tabla, incluyendo el desafío de la parte 6:

| | |
|---|---|
| **Herramienta y versión** | *por ejemplo: Claude Code, noviembre de 2026* |
| **Para qué la usaste** | *script, guion de SLURM, depuración…* |
| **Prompts principales** | *copia los dos o tres más importantes* |
| **Qué verificaste tú** | *cómo comprobaste que el resultado era correcto* |
| **Qué corregiste** | *errores de la IA que encontraste y cómo los arreglaste* |

## Lista de control antes de entregar

- [ ] *Reiniciar y ejecutar todas* corre sin errores y las autoverificaciones muestran ✓.
- [ ] Registraste versiones y semilla (la celda de la parte 1 las imprime).
- [ ] Respondiste la pregunta 5, la reflexión y la declaración de IA, y pegaste el script y el guion de SLURM.
- [ ] El archivo se llama `D3_Apellido_Nombre.ipynb` y lo subiste al aula virtual antes del jueves 19/11.

**Rúbrica (10 puntos):** reproducibilidad (versiones y semillas) 1.5 · flujo de Spark implementado 3 · validación honesta y comparación con cifras 2 · desafío con IA verificado y declarado 2 · reflexión sobre 100 veces más datos 1.5.